# Задание 3

## Уравнение:
$$u` = x + u^{3}$$

## Начальные условия
$$u(0) = 0$$

In [50]:
import math
import matplotlib.pyplot as plt
import numpy as np

In [51]:
def euler(x0, y0, h, n, func):
    res = []
    x, y = x0, y0

    # res.append(y)
    for i in range(n):
        y += h * func(x, y)
        x += h
        res.append(y)
    return res

def euler_one_step(x, U, step):
    f_val = step * x + U * step * U * U
    big_result = U + f_val
    return big_result


def find_const(x, U):
    return 1 / (2 * U * U) + x

def adaptive_euler(eps=1e-4, h_init=1, x_max_limit=1.69):
    x = 0.0
    U = 0.0
    h = h_init
    h_limit = 1e-10

    f = 1
    f10 = 0
    U_prev = 0.0

    while True:
        U_big = euler_one_step(x, U, h)

        half = h / 2.0
        U_half_1 = euler_one_step(x, U, half)
        U_small = euler_one_step(x + half, U_half_1, half)

        denominator = max(1.0, abs(U_small))
        local_error = abs(U_big - U_small) / denominator

        if local_error > eps:
            h /= 2.0
            if h < h_limit:
                print("Шаг стал слишком малым")
                return x, U_small, h
            continue
        else:
            x += h
            U = U_small

            if f10 < 10 and U > 10 ** f10:
                print(f" x={x:.20E}\n c={find_const(x, U):.80E}")
                f10 += 1

            if U > 10 ** (f * 10):
                print(f"x={x:.20E}, U={U:.8E}, h={h:.8E}")
                print(f"c={find_const(x, U):.80E}")
                f += 1

            if math.isinf(U):
                return x - h, U_prev, h * 2

            if x > x_max_limit:
                return x_max_limit, U, h

            U_prev = U

def find_max_x_with_tolerance(func, x0, u0, h, eps):
    x_current = x0
    u_big = u0
    u_small = u0

    while True:
        f_val = func(x_current, u_big)
        u_big_next = u_big + h * f_val
        x_next = x_current + h

        half = h / 2.0
        u_temp = u_small + half * func(x_current, u_small)
        x_mid = x_current + half
        u_temp += half * func(x_mid, u_temp)

        u_small_next = u_temp

        denom = max(1.0, abs(u_small_next))
        rel_err = abs(u_big_next - u_small_next) / denom

        if rel_err > eps:
            return x_current

        x_current = x_next
        u_big = u_big_next
        u_small = u_small_next


def build_x_array(x_start, x_end, h):
    return np.arange(x_start, x_end + h, h).tolist()


def runge_kutta4(x0, y0, h, n, func):
    res = []
    x, y = x0, y0

    # res.append(y)
    for i in range(n):
        k1 = func(x, y)
        k2 = func(x + h / 2.0, y + (h / 2.0) * k1)
        k3 = func(x + h / 2.0, y + (h / 2.0) * k2)
        k4 = func(x + h, y + h * k3)

        delta = (k1 + 2.0 * k2 + 2.0 * k3 + k4) / 6.0

        y += h * delta
        x += h
        res.append(y)

    return res

def picard1(x):
    return x ** 2 / 2

def picard2(x):
    return x ** 7 / 56 + x ** 2 / 2

def picard3(x):
    return (x ** 22 / 3863552
            + 3 * x ** 17 / 106624
            + x ** 12 / 896
            + x ** 7 / 56
            + x ** 2 / 2)

def picard4(x):
    return (x ** 67 / 3863981943017739124736
            + 9 * x ** 62 / 98677964914244452352
            + 1081 * x ** 57 / 73440052228804050944
            + 76623 * x ** 52 / 53388985337387155456
            + 310503 * x ** 47 / 3244062457475366912
            + (1069 * x ** 42) / 224099368435712
            + (790667 * x ** 37) / 4145838316060672
            + (11871 * x ** 32) / 1883187970048
            + (361 * x ** 27) / 2101772288
            + (47 * x ** 22) / 11941888
            + (33 * x ** 17) / 426496
            + (x ** 12) / 896
            + (x ** 7) / 56
            + (x ** 2) / 2)


def func(x, y):
    return x + y ** 3


def table_print(x, y1, y2, y3, y4, y5, y6):
    print("---------------------------------------------------------------------------------------------------------")
    print("|      x      |    Пикар 1   |    Пикар 2   |    Пикар 3   |    Пикар 4   |     Эйлер    | Рунге-Кутта4 |")
    print("---------------------------------------------------------------------------------------------------------")
    for i in range(len(x)):
        print("|{:^14.5f}|{:^14.5f}|{:^14.5f}|{:^14.5f}|{:^14.5f}|{:^14.5f}|{:^14.5f}|".format(
            x[i], y1[i], y2[i], y3[i], y4[i], y5[i], y6[i]))
    print("---------------------------------------------------------------------------------------------------------")


In [52]:
h = 1e-5
eps = 1e-10

x0 = 0.
u0 = 0.

print(adaptive_euler(1e-10))
# x_max = find_max_x_with_tolerance(func, x0, u0, h, eps)
x_max = 1.65
print(x_max)
# x_max = 1.8
x_arr = build_x_array(x0, x_max, h)

n = len(x_arr) - 1

X = []
Y_first = []
Y_second = []
Y_third = []
Y_forth = []
Y_euler = euler(0, 0, h, n, func)
Y_runge_kutta = runge_kutta4(0, 0, h, n, func)
for i in range(n):
    x = x0 + i * h
    X.append(x)
    Y_first.append(picard1(x))
    Y_second.append(picard2(x))
    Y_third.append(picard3(x))
    Y_forth.append(picard4(x))
table_print(X, Y_first, Y_second, Y_third, Y_forth, Y_euler, Y_runge_kutta)
plt.plot(X, Y_euler, label='Эйлер')
plt.plot(X, Y_runge_kutta, label='Рунге-Кутта 4')
plt.plot(X, Y_first, label='1 приближение')
plt.plot(X, Y_second, label='2 приближение')
plt.plot(X, Y_third, label='3 приближение')
plt.plot(X, Y_forth, label='4 приближение')
plt.legend()
plt.show()


 x=1.30339431762695312500E+00
 c=1.80338656635790783866468700580298900604248046875000000000000000000000000000000000E+00
 x=1.64225691556930541992E+00
 c=1.64725691351034053688806579884840175509452819824218750000000000000000000000000000E+00
 x=1.64720365963876247406E+00
 c=1.64725365948406543559201509197009727358818054199218750000000000000000000000000000E+00
Шаг стал слишком малым
(1.647248618886806, 314.9474328089159, 5.820766091346741e-11)
1.65


OverflowError: (34, 'Numerical result out of range')